# Appendix B: ICO constraints and certificates

Load the saved exact CG matrix, derive and print the 166 affine constraints,
then use that same symbolic solution to verify the certificates in Eq. (S28).
Extract the weighted objective blocks from the stored performance operator;
only the certificate matrices $Q_k^{(r)}$ are entered manually.
The calculation runs from top to bottom without reading the constraint TXT
or executing another notebook. Required inputs: `Gmatrix_exact.npy` and
`Omega_Tilde_Symbolic_d2_N3.pkl`.


In [1]:
import pickle
import numpy as np
import sympy as sp

BLOCK_DIMENSIONS = [(4,1), (6,3), (2,5), (6,3), (9,9),
                    (3,15), (2,5), (3,15), (1,25)]
G = sp.SparseMatrix(np.load("Gmatrix_exact.npy", allow_pickle=True))
assert G.H * G == sp.eye(256)
assert not G.atoms(sp.Float)
print("PASS: loaded exact orthogonal CG matrix.")


PASS: loaded exact orthogonal CG matrix.


## Construct the strategy

Each $H_k$ is Hermitian. Its diagonal entries and the real and imaginary parts
above the diagonal give 196 real coordinates. Form
$C=G[\bigoplus_k(I_{d_k}\otimes H_k)]G^\dagger$ in the port order
$(P,I_1,O_1,I_2,O_2,I_3,O_3,F)$.


In [2]:
coordinates, blocks = [], []
for k, (m, d) in enumerate(BLOCK_DIMENSIONS):
    H = sp.zeros(m)
    for i in range(m):
        for j in range(i, m):
            real = sp.Symbol(f"H{k}_R_{i}_{j}", real=True)
            coordinates.append(real)
            H[i,j] = H[j,i] = real
            if i != j:
                imaginary = sp.Symbol(f"H{k}_I_{i}_{j}", real=True)
                coordinates.append(imaginary)
                H[i,j] += sp.I * imaginary
                H[j,i] -= sp.I * imaginary
    blocks.append(H)

schur_blocks = sp.diag(*[
    sp.kronecker_product(sp.eye(d), H)
    for H, (m, d) in zip(blocks, BLOCK_DIMENSIONS)
])
C = (G * sp.SparseMatrix(schur_blocks) * G.H).applyfunc(sp.expand)
C_array = np.array(C.tolist(), dtype=object)
print("Real Hermitian coordinates:", len(coordinates))


Real Hermitian coordinates: 196


## Trace-and-replace operation

For a set of ports $X$, Eq. (2) uses the normalized operation
${}_X M=(I_X/d_X)\otimes\operatorname{Tr}_X M$, with the original port order
restored. The function below implements this operation on exact symbolic arrays.


In [3]:
def trace_and_replace(matrix, ports, n_ports):
    ports = sorted(ports)
    keep = [p for p in range(n_ports) if p not in ports]
    axes = keep + ports + [p+n_ports for p in keep] + [p+n_ports for p in ports]
    d_keep, d_trace = 2**len(keep), 2**len(ports)
    tensor = matrix.reshape([2]*(2*n_ports)).transpose(axes)
    tensor = tensor.reshape(d_keep, d_trace, d_keep, d_trace)
    reduced = np.trace(tensor, axis1=1, axis2=3)
    replaced = np.kron(reduced, np.eye(d_trace, dtype=object))
    replaced = replaced * sp.Rational(1, d_trace)
    return replaced.reshape([2]*(2*n_ports)).transpose(np.argsort(axes)).reshape(matrix.shape)


P, I1, O1, I2, O2, I3, O3, F = range(8)


def R(*ports):
    # R(*X) is exactly the paper's normalized trace-and-replace {}_X C.
    return trace_and_replace(C_array, ports, 8)


def slot_permutation_matrix(order):
    indices = np.arange(256).reshape([2]*8).transpose(order).ravel()
    return sp.SparseMatrix(256, 256, {(i, int(j)): 1 for i,j in enumerate(indices)})


## The seven ICO equalities in Eq. (2)

For $n=3$, the seven nonempty subsets are
$\{1\},\{2\},\{3\},\{1,2\},\{1,3\},\{2,3\},\{1,2,3\}$.
Every named expression below is set equal to zero. `R(I2,O2,I3,O3,F)` means
${}_{I_2O_2I_3O_3F}C$, using exactly the notation of the paper.
For example, the first line is

```math
{}_{I_2O_2I_3O_3F}C-{}_{O_1I_2O_2I_3O_3F}C=0.
```

The conditions are written individually, as in the original ICO program.


In [4]:
ico_constraints = {}

# Eq. (2): one active slot.
ico_constraints["S={1}"] = R(I2,O2,I3,O3,F) - R(O1,I2,O2,I3,O3,F)
ico_constraints["S={2}"] = R(I1,O1,I3,O3,F) - R(I1,O1,O2,I3,O3,F)
ico_constraints["S={3}"] = R(I1,O1,I2,O2,F) - R(I1,O1,I2,O2,O3,F)

# Eq. (2): two active slots.
ico_constraints["S={1,2}"] = (
    R(I3,O3,F) - R(O1,I3,O3,F) - R(O2,I3,O3,F) + R(O1,O2,I3,O3,F)
)
ico_constraints["S={1,3}"] = (
    R(I2,O2,F) - R(O1,I2,O2,F) - R(O3,I2,O2,F) + R(O1,O3,I2,O2,F)
)
ico_constraints["S={2,3}"] = (
    R(I1,O1,F) - R(O2,I1,O1,F) - R(O3,I1,O1,F) + R(O2,O3,I1,O1,F)
)

# Eq. (2): all three slots.
ico_constraints["S={1,2,3}"] = (
    R(F) - R(O1,F) - R(O2,F) - R(O3,F)
    + R(O1,O2,F) + R(O1,O3,F) + R(O2,O3,F) - R(O1,O2,O3,F)
)
print("ICO equalities imposed:", ", ".join(ico_constraints))


ICO equalities imposed: S={1}, S={2}, S={3}, S={1,2}, S={1,3}, S={2,3}, S={1,2,3}


## Normalization and slot-permutation symmetry

The other two equalities in Eq. (2) are imposed separately:

```math
{}_{I^3O^3F}C={}_{PI^3O^3F}C,\qquad
\operatorname{Tr}C=\sum_k d_k\operatorname{Tr}H_k=16.
```

Appendix B also uses $C=P_{12}CP_{12}^{\dagger}$ and
$C=P_{23}CP_{23}^{\dagger}$. These two slot transpositions generate $S_3$.
The positivity condition $C\succeq0$, equivalently $H_k\succeq0$, remains
a separate feasibility condition; here we solve the linear equalities.


In [5]:
input_normalization = R(I1,O1,I2,O2,I3,O3,F) - R(P,I1,O1,I2,O2,I3,O3,F)
trace_normalization = sum(d*sp.trace(H) for H,(m,d) in zip(blocks,BLOCK_DIMENSIONS)) - 16

P12 = slot_permutation_matrix([0,3,4,1,2,5,6,7])
P23 = slot_permutation_matrix([0,1,2,5,6,3,4,7])
swap12_constraint = np.array((C - P12*C*P12.H).tolist(), dtype=object)
swap23_constraint = np.array((C - P23*C*P23.H).tolist(), dtype=object)

matrix_equations = list(ico_constraints.values()) + [
    input_normalization, swap12_constraint, swap23_constraint
]
print("Also imposed: input normalization, Tr(C)=16, swap(1,2), swap(2,3).")


Also imposed: input normalization, Tr(C)=16, swap(1,2), swap(2,3).


## Combine the physical constraints into scalar equations

All the matrix equalities above hold simultaneously. Take their real and
imaginary matrix elements, remove zero or repeated expressions, and include
the scalar trace condition. `sp.Eq(expression, 0)` explicitly sets each
expression equal to zero.


In [6]:
expressions = set()
for matrix in matrix_equations:
    for entry in set(matrix.flat):
        if entry == 0:
            continue
        for part in sp.expand(entry).as_real_imag():
            part = sp.expand(part)
            if part != 0:
                expressions.add(part)
expressions.add(trace_normalization)
expressions = sorted(expressions, key=str)
equations = [sp.Eq(expression, 0) for expression in expressions]
print("Distinct nonzero scalar equations:", len(equations))


Distinct nonzero scalar equations: 2498


## Solve the simultaneous equations

Solve the physical equalities together with SymPy. All coordinates are real
variables; the coordinates not solved for remain free parameters.
For this linear system, `sp.solve` returns the substitution dictionary directly.


In [7]:
substitutions = sp.solve(equations, coordinates)
free_coordinates = [variable for variable in coordinates if variable not in substitutions]
print("Solved coordinates:", len(substitutions))
print("Free parameters:", len(free_coordinates))


Solved coordinates: 166
Free parameters: 30


## Check and print the constraints

Substitute the affine solution into every generated scalar equation, then
print the 166 solved equalities below. They can be copied in the same way
as the original public constraint file.


In [8]:
assert len(substitutions) == 166 and len(free_coordinates) == 30
assert all(sp.simplify(sp.expand(expression.xreplace(substitutions))) == 0 for expression in expressions)
print("PASS: every generated equation is satisfied exactly.")
for variable, expression in substitutions.items():
    print(f"{variable} = {expression}")


PASS: every generated equation is satisfied exactly.
H0_I_0_1 = 0
H0_I_0_2 = 0
H0_I_0_3 = 0
H0_I_1_2 = 0
H0_I_1_3 = 0
H0_I_2_3 = 0
H0_R_0_0 = -3*H1_R_4_4 - 3*sqrt(6)*H3_R_3_4/2 - 6*sqrt(2)*H3_R_3_5 - 33*H3_R_5_5/16 - 9*sqrt(6)*H4_R_4_6/2 - 18*sqrt(2)*H4_R_4_7 + 9*sqrt(2)*H4_R_5_8/4 - 99*H4_R_7_7/16 - 117*H4_R_8_8/32 + 15*sqrt(2)*H5_R_1_2/4 - 195*H5_R_2_2/32 + 25*H6_R_1_1/16 + 75*H7_R_1_1/16 - 195*H7_R_2_2/32 - 325*H8_R_0_0/32 + 2
H0_R_0_1 = 3*H1_R_3_4 - 3*sqrt(6)*H3_R_3_5/2 + 3*sqrt(3)*H3_R_5_5/16 - 9*sqrt(6)*H4_R_4_7/2 + 9*sqrt(6)*H4_R_5_8/4 + 9*sqrt(3)*H4_R_7_7/16 - 9*sqrt(3)*H4_R_8_8/32 + 15*sqrt(6)*H5_R_1_2/4 - 15*sqrt(3)*H5_R_2_2/32 + 5*sqrt(3)*H6_R_1_1/16 + 15*sqrt(3)*H7_R_1_1/16 - 15*sqrt(3)*H7_R_2_2/32 - 25*sqrt(3)*H8_R_0_0/32
H0_R_0_2 = 3*H1_R_3_4 - 3*sqrt(6)*H3_R_3_5/2 + 3*sqrt(3)*H3_R_5_5/16 - 9*sqrt(6)*H4_R_4_7/2 + 9*sqrt(6)*H4_R_5_8/4 + 9*sqrt(3)*H4_R_7_7/16 - 9*sqrt(3)*H4_R_8_8/32 + 15*sqrt(6)*H5_R_1_2/4 - 15*sqrt(3)*H5_R_2_2/32 + 5*sqrt(3)*H6_R_1_1/16 + 15*sqrt(3)*H7_R_1

## Substitute the 166 constraints into the same blocks

Continue with `blocks` and `substitutions` from above. The 30 free parameters
remain symbolic throughout the certificate calculation.


In [9]:
affine_blocks = [
    H.xreplace(substitutions).applyfunc(sp.expand)
    for H in blocks
]

assert (
    len(coordinates) == 196
    and len(substitutions) == 166
    and (len(free_coordinates) == 30)
)
assert all(
    (rhs.free_symbols <= set(free_coordinates) for rhs in substitutions.values())
)
assert all(
    (
        sp.Poly(rhs, *free_coordinates).total_degree() <= 1
        for rhs in substitutions.values()
    )
)
assert all((block == block.H for block in affine_blocks))
assert all(
    (
        sp.expand((lhs - rhs).xreplace(substitutions)) == 0
        for lhs, rhs in substitutions.items()
    )
)
assert (
    sp.expand(
        sum(
            (
                d * sp.trace(affine_blocks[k])
                for k, (m, d) in enumerate(BLOCK_DIMENSIONS)
            )
        )
        - 16
    )
    == 0
)


## Extract the weighted objective blocks from the stored operator

Load `Omega_Tilde_Symbolic_d2_N3.pkl`, which contains the exact performance
operator in the CG basis. Separate its cubic noise dependence into the four
noise-count terms $r=0,1,2,3$, then extract $d_k\Omega_k^{(r)}$ from each
repeated block $I_{d_k}\otimes\Omega_k^{(r)}$.

Check that the four terms reconstruct the stored operator and that the
extracted blocks reconstruct each complete term. All entries remain exact.


In [10]:
def expand_matrix(matrix):
    return matrix.applyfunc(sp.expand)


def assert_exact_matrix_equal(left, right, label):
    assert expand_matrix(left-right) == sp.zeros(*left.shape), label


with open("Omega_Tilde_Symbolic_d2_N3.pkl", "rb") as stream:
    performance_data = pickle.load(stream)
performance_operator = sp.SparseMatrix(performance_data["Omega_tilde"])
noise_parameter = performance_data["p_symbol"]
assert performance_operator.shape == (256, 256)
assert performance_operator.free_symbols == {noise_parameter}
assert not performance_operator.atoms(sp.Float)
noise_count_entries = [{} for _ in range(4)]
for (row, column), value in performance_operator.todok().items():
    polynomial = sp.Poly(sp.expand(value), noise_parameter)
    assert polynomial.degree() <= 3
    a0, a1, a2, a3 = [polynomial.nth(j) for j in range(4)]
    noise_count_coefficients = (
        a0,
        a0 + a1 / 3,
        a0 + 2 * a1 / 3 + a2 / 3,
        a0 + a1 + a2 + a3,
    )
    for r, coefficient in enumerate(noise_count_coefficients):
        coefficient = sp.expand(coefficient)
        if coefficient:
            noise_count_entries[r][row, column] = coefficient
noise_count_operators = [
    sp.SparseMatrix(256, 256, matrix_entries) for matrix_entries in noise_count_entries
]
noise_count_weights = (
    (1 - noise_parameter) ** 3,
    3 * noise_parameter * (1 - noise_parameter) ** 2,
    3 * noise_parameter**2 * (1 - noise_parameter),
    noise_parameter**3,
)
reconstructed_performance_operator = sum(
    (noise_count_weights[r] * noise_count_operators[r] for r in range(4)), sp.zeros(256)
)
assert_exact_matrix_equal(
    reconstructed_performance_operator,
    performance_operator,
    "complete noise-count reconstruction",
)
weighted_objective_blocks = {}
for r in range(4):
    weighted_objective_blocks[r] = {}
    sector_offset = 0
    for k, (m, d) in enumerate(BLOCK_DIMENSIONS):
        value = expand_matrix(
            d
            * noise_count_operators[r][
                sector_offset : sector_offset + m, sector_offset : sector_offset + m
            ]
        )
        weighted_objective_blocks[r][k] = value
        sector_offset += m * d
    reconstructed_block_operator = sp.diag(
        *[
            sp.kronecker_product(sp.eye(d), weighted_objective_blocks[r][k] / d)
            for k, (m, d) in enumerate(BLOCK_DIMENSIONS)
        ]
    )
    assert_exact_matrix_equal(
        noise_count_operators[r], reconstructed_block_operator, f"complete r={r} block form"
    )
assert_exact_matrix_equal(noise_count_operators[3], sp.eye(256) / 64, "r=3 term")

print("PASS: objective blocks extracted from .pkl; exact reconstruction and block normalization verified.")


PASS: objective blocks extracted from .pkl; exact reconstruction and block normalization verified.


## Certificate matrix inputs

For each $r=1,2$, the certificate has nine blocks $Q_k^{(r)}$, $k=0,\ldots,8$.
Enter these 18 certificate matrices below, using the exact entries displayed
in `Certificates.md`, with $s=\sqrt2$ and $t=\sqrt3$.
The weighted objective blocks have already been extracted from the `.pkl`.


In [11]:
s, t = sp.sqrt(2), sp.sqrt(3)
sqrt2, sqrt3 = s, t
certificate_blocks = {1: {}, 2: {}}


### One depolarizing term: $r=1$

Enter the nine certificate blocks $Q_k^{(1)}$ in order $k=0,\ldots,8$.


In [12]:
certificate_blocks[1][0] = sp.Rational(1, 72) * sp.Matrix([
    [s + 1, 0, 0, s + 1],
    [0, s + 1, -s - 1, 0],
    [0, -s - 1, s + 1, 0],
    [s + 1, 0, 0, s + 1],
])

certificate_blocks[1][1] = sp.Rational(1, 48) * sp.Matrix([
    [2*s + 2, -2*t, -s*t, -2*t, 2*s - 2, s],
    [-2*t, 2*s + 6, -s, 6 - 2*s, 2*t, -s*t],
    [-s*t, -s, 2*s, s, -s*t, 0],
    [-2*t, 6 - 2*s, s, 2*s + 6, 2*t, s*t],
    [2*s - 2, 2*t, -s*t, 2*t, 2*s + 2, s],
    [s, -s*t, 0, s*t, s, 2*s],
])

certificate_blocks[1][2] = sp.Rational(5, 72) * sp.Matrix([
    [s + 2, 0],
    [0, s + 2],
])

certificate_blocks[1][3] = sp.Rational(1, 432) * sp.Matrix([
    [18*s + 18, 6*t, -18*t, 20 - 6*s, -9*s*t, 12 - 29*s],
    [6*t, 18*s + 6, 6*s - 32, -14*t, 23*s - 12, -7*s*t],
    [-18*t, 6*s - 32, 18*s + 54, -6*t, -9*s, 29*s*t - 12*t],
    [20 - 6*s, -14*t, -6*t, 18*s + 34, -23*s*t + 12*t, -7*s],
    [-9*s*t, 23*s - 12, -9*s, -23*s*t + 12*t, 18*s + 36, 0],
    [12 - 29*s, -7*s*t, 29*s*t - 12*t, -7*s, 0, 18*s + 68],
])

certificate_blocks[1][4] = sp.Rational(1, 144) * sp.Matrix([
    [18*s + 18, -12*t, -9*s*t, -12*t, 32 - 6*s, -3*s, 3*s*t, 12 - 23*s, -3],
    [-12*t, 18*s + 42, -9*s, 6*s - 8, -4*t, -s*t, 17*s - 12, -11*s*t, -t],
    [-9*s*t, -9*s, 18*s + 6, 21*s, -s*t, 0, 3, 17*t, 0],
    [-12*t, 6*s - 8, 21*s, 18*s + 42, 12*t, 9*s*t, 3*s, 23*s*t - 12*t, 3*t],
    [32 - 6*s, -4*t, -s*t, 12*t, 18*s + 50, -7*s, -17*s*t + 12*t, -11*s, -1],
    [-3*s, -s*t, 0, 9*s*t, -7*s, 18*s + 6, -3*t, 17, 0],
    [3*s*t, 17*s - 12, 3, 3*s, -17*s*t + 12*t, -3*t, 18*s + 66, 0, 0],
    [12 - 23*s, -11*s*t, 17*t, 23*s*t - 12*t, -11*s, 17, 0, 18*s + 34, 16*s],
    [-3, -t, 0, 3*t, -1, 0, 0, 16*s, 18*s + 6],
])

certificate_blocks[1][5] = sp.Rational(5, 216) * sp.Matrix([
    [9*s + 21, 8*t, 4*s*t],
    [8*t, 9*s + 5, 4*s],
    [4*s*t, 4*s, 9*s + 1],
])

certificate_blocks[1][6] = sp.Rational(5, 72) * sp.Matrix([
    [s + 4, 0],
    [0, s + 4],
])

certificate_blocks[1][7] = sp.Rational(5, 24) * sp.Matrix([
    [s + 3, 0, 0],
    [0, s + 3, 0],
    [0, 0, s + 1],
])

certificate_blocks[1][8] = sp.Rational(25, 72) * sp.Matrix([
    [s - 1],
])


### Two depolarizing terms: $r=2$

Enter the nine certificate blocks $Q_k^{(2)}$ in order $k=0,\ldots,8$.


In [13]:
certificate_blocks[2][0] = sp.Rational(1, 1152) * sp.Matrix([
    [8*s + 5, 3*t, 3*t, 8*s - 13],
    [3*t, 8*s - 1, 7 - 8*s, -3*t],
    [3*t, 7 - 8*s, 8*s - 1, -3*t],
    [8*s - 13, -3*t, -3*t, 8*s + 5],
])

certificate_blocks[2][1] = sp.Rational(1, 384) * sp.Matrix([
    [8*s + 5, -5*t, -4*s*t, -5*t, 8*s - 5, 4*s],
    [-5*t, 8*s + 15, -4*s, 15 - 8*s, 5*t, -4*s*t],
    [-4*s*t, -4*s, 8*s, 4*s, -4*s*t, 0],
    [-5*t, 15 - 8*s, 4*s, 8*s + 15, 5*t, 4*s*t],
    [8*s - 5, 5*t, -4*s*t, 5*t, 8*s + 5, 4*s],
    [4*s, -4*s*t, 0, 4*s*t, 4*s, 8*s],
])

certificate_blocks[2][2] = sp.Rational(5, 144) * sp.Matrix([
    [s + 2, 0],
    [0, s + 2],
])

certificate_blocks[2][3] = sp.Rational(1, 17280) * sp.Matrix([
    [360*s + 225, 195*t, -225*t, 3 - 120*s, -180*s*t, 240 - 228*s],
    [195*t, 360*s - 165, 120*s - 393, -215*t, 168*s - 240, -40*s*t],
    [-225*t, 120*s - 393, 360*s + 675, -195*t, -180*s, 228*s*t - 240*t],
    [3 - 120*s, -215*t, -195*t, 360*s + 265, -168*s*t + 240*t, -40*s],
    [-180*s*t, 168*s - 240, -180*s, -168*s*t + 240*t, 360*s + 180, 0],
    [240 - 228*s, -40*s*t, 228*s*t - 240*t, -40*s, 0, 360*s + 980],
])

certificate_blocks[2][4] = sp.Rational(1, 5760) * sp.Matrix([
    [360*s + 225, -165*t, -180*s*t, -105*t, 363 - 120*s, 180*s, 60*s*t, 240 - 228*s, 0],
    [-165*t, 360*s + 555, -180*s, 120*s - 33, -15*t, -20*s*t, 168*s - 240, -120*s*t, 160*t],
    [-180*s*t, -180*s, 360*s + 60, 180*s, -20*s*t, -100*t, 0, 160*t, -50*s*t],
    [-105*t, 120*s - 33, 180*s, 360*s + 435, 165*t, 180*s*t, 60*s, 228*s*t - 240*t, 0],
    [363 - 120*s, -15*t, -20*s*t, 165*t, 360*s + 585, -140*s, -168*s*t + 240*t, -120*s, 160],
    [180*s, -20*s*t, -100*t, 180*s*t, -140*s, 360*s + 260, 0, 160, -50*s],
    [60*s*t, 168*s - 240, 0, 60*s, -168*s*t + 240*t, 0, 360*s + 780, 0, 0],
    [240 - 228*s, -120*s*t, 160*t, 228*s*t - 240*t, -120*s, 160, 0, 360*s + 300, 320*s],
    [0, 160*t, -50*s*t, 0, 160, -50*s, 0, 320*s, 360*s + 310],
])

certificate_blocks[2][5] = sp.Rational(5, 1728) * sp.Matrix([
    [36*s + 78, 22*t, 11*s*t],
    [22*t, 36*s + 34, 11*s],
    [11*s*t, 11*s, 36*s + 23],
])

certificate_blocks[2][6] = sp.Rational(5, 288) * sp.Matrix([
    [2*s + 5, 0],
    [0, 2*s + 5],
])

certificate_blocks[2][7] = sp.Rational(5, 192) * sp.Matrix([
    [4*s + 6, 0, 0],
    [0, 4*s + 6, 0],
    [0, 0, 4*s + 7],
])

certificate_blocks[2][8] = sp.Rational(25, 576) * sp.Matrix([
    [4*s - 1],
])


assert all(
    set(matrix_family[r]) == set(range(9))
    for matrix_family in (weighted_objective_blocks, certificate_blocks)
    for r in (1, 2)
)

for r in (1, 2):
    for k in range(9):
        assert (
            weighted_objective_blocks[r][k].shape
            == certificate_blocks[r][k].shape
            == (BLOCK_DIMENSIONS[k][0], BLOCK_DIMENSIONS[k][0])
        )
        assert weighted_objective_blocks[r][k] == weighted_objective_blocks[r][k].H
        assert certificate_blocks[r][k] == certificate_blocks[r][k].H


## Check the two certificate identities

For $r=1,2$, substitute the 166 constraints into Eq. (S28) and calculate

```math
\beta_r-\sum_{k=0}^8\operatorname{Tr}
\left[(d_k\Omega_k^{(r)}+Q_k^{(r)})H_k\right]=0.
```

All 30 free parameters remain symbolic, so a zero residual checks the whole
affine space.


In [14]:
noise_count_bounds = {1: (21 + 8 * sqrt2) / 36, 2: (27 + 8 * sqrt2) / 72}

for r in (1, 2):
    certificate_sum = 0
    for k in range(9):
        certificate_sum += sp.trace(
            (weighted_objective_blocks[r][k] + certificate_blocks[r][k])
            * affine_blocks[k]
        )
    residual = sp.expand(noise_count_bounds[r] - certificate_sum)
    assert residual == 0
    print(f"PASS: Eq. (S28), r={r}, residual = {residual}.")


PASS: Eq. (S28), r=1, residual = 0.


PASS: Eq. (S28), r=2, residual = 0.


## Positive certificates

Check $Q_k^{(r)}\succeq0$ exactly using Gram decompositions and leading principal minors.


In [15]:
assert all(weight.is_positive is True for weight in (1+sqrt2, 2*sqrt2-1, sqrt2))


def outer_product(vector):
    vector = sp.Matrix(vector)
    return vector * vector.T

# Singular blocks use Gram decompositions; the other blocks use Sylvester's criterion.
gram_a = sp.Matrix([1, 0, 0, 1])
gram_b = sp.Matrix([0, 1, -1, 0])
gram_u = sp.Matrix([sqrt2 * sqrt3, sqrt2, sqrt2, -sqrt2 * sqrt3]) / 4
gram_w1 = sp.Matrix([sqrt2, sqrt2 * sqrt3, -sqrt2 * sqrt3, sqrt2]) / 4
gram_w2 = sp.Matrix([-sqrt2 * sqrt3, sqrt2, -sqrt2, -sqrt2 * sqrt3]) / 4
gram_p = sp.Matrix([-sqrt2, sqrt2 * sqrt3, 0, sqrt2 * sqrt3, sqrt2, 0]) / 4
gram_v1 = sp.Matrix([-1, -sqrt3, sqrt3, sqrt3, -1, 1]) / 2
gram_v2 = sp.Matrix([sqrt3, -1, -1, 1, sqrt3, sqrt3]) / 2
assert_exact_matrix_equal(
    certificate_blocks[1][0],
    (1 + sqrt2) / 72 * (outer_product(gram_a) + outer_product(gram_b)),
    "Gram Q[1][0]",
)
assert_exact_matrix_equal(
    certificate_blocks[2][0],
    sp.Rational(1, 48) * outer_product(gram_u)
    + (2 * sqrt2 - 1) / 144 * (outer_product(gram_w1) + outer_product(gram_w2)),
    "Gram Q[2][0]",
)
assert_exact_matrix_equal(
    certificate_blocks[1][1],
    sp.Rational(1, 3) * outer_product(gram_p)
    + sqrt2 / 24 * (outer_product(gram_v1) + outer_product(gram_v2)),
    "Gram Q[1][1]",
)
assert_exact_matrix_equal(
    certificate_blocks[2][1],
    sp.Rational(5, 48) * outer_product(gram_p)
    + sqrt2 / 48 * (outer_product(gram_v1) + outer_product(gram_v2)),
    "Gram Q[2][1]",
)
for r in (1, 2):
    for k in range(2, 9):
        for minor_order in range(1, certificate_blocks[r][k].rows + 1):
            principal_minor = certificate_blocks[r][k][:minor_order, :minor_order].det()
            assert sp.simplify(principal_minor).is_positive is True

print("PASS: all 18 certificate blocks are positive semidefinite.")


PASS: all 18 certificate blocks are positive semidefinite.
